# TP : Simulation Quantique et Évolution Temporelle Moléculaire

**Objectifs de ce TP :**
1. Définir un système moléculaire simple ($H_2$) avec la base STO-3G.
2. Appliquer les transformations fermions-qubits (Jordan-Wigner, Bravyi-Kitaev, Parité).
3. Calculer les énergies propres de l'Hamiltonien via diagonalisation exacte (Numpy).
4. Simuler l'évolution temporelle (matrice unitaire $e^{-iHt}$) avec différentes méthodes : Trotter, qDRIFT (aléatoire) et une méthode Hybride.
5. Comparer quantitativement ces méthodes.
6. Intégrer cet opérateur unitaire dans un circuit quantique avec un qubit de contrôle.
7. Créer une fonction génératrice pour évaluer l'opérateur de Pauli selon différentes distances interatomiques.


In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Qiskit et Qiskit Nature
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper, BravyiKitaevMapper, ParityMapper

## 1. Fonction Génératrice du Système Moléculaire ($H_N$)

Afin de pouvoir tester différentes molécules au cours du TP, nous allons directement créer une fonction. Elle utilisera `PySCFDriver` pour définir une chaîne d'atomes d'hydrogène ($H_N$) espacés d'une distance régulière dans la base `STO-3G`.

**Indices de réalisation :**

*   La chaîne de caractères décrivant la géométrie doit avoir le format : `"H 0 0 0; H 0 0 distance; H 0 0 2*distance..."`. Utilisez une boucle ou une compréhension de liste Python combinée avec `"; ".join(...)` pour générer cela dynamiquement en fonction de `num_atoms`.
*   Initialisez le `PySCFDriver` avec cette géométrie.
*   Extrayez le problème, puis l'Hamiltonien fermionique.
*   Appliquez le `mapper` passé en argument pour retourner l'opérateur de Pauli.

In [ ]:
def get_h_chain_pauli_op(num_atoms: int, distance: float, mapper) -> SparsePauliOp:
    """Génère l'opérateur qubit pour une chaîne d'atomes d'hydrogène."""
    
    # TODO: Construire la chaîne de géométrie atomique
    # atom_string = ...
    
    # TODO: Initialisez le driver PySCF avec les bons paramètres
    # driver = PySCFDriver(atom=atom_string, ...)
    
    # TODO: Récupérez le problème moléculaire et l'opérateur de seconde quantification
    # problem = ...
    # hamiltonian = ...
    
    # TODO: Appliquez le mapper et retournez l'opérateur qubit
    # return mapper.map(hamiltonian)
    pass

## 2. Transformations Fermions-Qubits et Comparaison

Les opérateurs fermioniques doivent être convertis en opérateurs de Pauli (qubits). Maintenant que notre fonction est prête, comparons les trois "mappers" sur la molécule $H_2$, $H_3$ $H_4$ et $H_5$.

**Indices de réalisation :**

*   Instanciez les trois mappers : `JordanWignerMapper()`, `BravyiKitaevMapper()`, et `ParityMapper()`.
*   Utilisez `get_h_chain_pauli_op` pour générer les opérateurs de $H_2$ (distance 0.735) pour chaque mapper.
*   Observez les différences dans le nombre de qubits (`op.num_qubits`) et la taille de l'opérateur (nombre de termes de Pauli avec `len(op)`) selon le mapper et la molécule.

In [ ]:
bond_distance = 0.735 # Angströms

# TODO: Instanciez les mappers
# jw_mapper = ...
# bk_mapper = ...
# parity_mapper = ...

# TODO: Générez les opérateurs pour H_n
# op_jw_h2 = get_h_chain_pauli_op(2, bond_distance, jw_mapper)
# op_bk_h2 = get_h_chain_pauli_op(2, bond_distance, bk_mapper)
# ...

# TODO: Affichez le nombre de qubits et de termes pour comparer

## 3. Diagonalisation Exacte (Numpy)

Avant de simuler l'évolution temporelle, choisissons un opérateur et calculons ses énergies propres exactes pour avoir une valeur de référence.

**Indices de réalisation :**

*   Choisissez l'opérateur généré avec le `ParityMapper` (souvent le plus économe en qubits) et assignez-le à la variable `qubit_op`.
*   Utilisez la méthode `.to_matrix()` de cet opérateur.
*   Utilisez `scipy.linalg.eigh()` pour extraire les valeurs propres. L'énergie fondamentale est la plus petite valeur.

In [ ]:
# TODO: Sélectionnez votre opérateur cible
# qubit_op = ...

# TODO: Convertissez l'opérateur qubit en matrice
# H_matrix = ...

# TODO: Calculez les valeurs propres et affichez l'énergie fondamentale
# valeurs_propres, vecteurs_propres = ...
# E0 = ...

## 4. Simulation de l'évolution temporelle ($e^{-iHt}$)

Nous cherchons à simuler l'opérateur d'évolution $U(t) = e^{-iHt}$. Vous allez comparer la matrice exacte à trois méthodes : Trotter, qDRIFT et une méthode Hybride.

**La méthode Hybride :**
L'Hamiltonien se décompose en $H = \sum c_k P_k$. La méthode hybride sépare ces termes selon un `threshold` (seuil) défini sur la valeur absolue du coefficient $\vert c_k \vert$ :

*   **Termes majeurs (**$\vert c_k \vert \ge \text{threshold}$**) :** Ils dictent la dynamique principale. On les simule de manière déterministe via la méthode de **Trotter**.
*   **Termes mineurs (**$\vert c_k \vert < \text{threshold}$**) :** Ils sont nombreux mais ont un faible impact individuel. On les simule de manière stochastique via **qDRIFT**.

*Conditions de performance :* Trotter est très précis mais la profondeur du circuit explose si le nombre de termes est grand. qDRIFT a une profondeur indépendante du nombre de termes, mais introduit du bruit sur les gros coefficients. Le `threshold` optimise le compromis entre erreur algorithmique et coût matériel.

**Indices de réalisation :**

*   Utilisez `qubit_op.paulis` et `qubit_op.coeffs` pour extraire les termes et coefficients.
*   **Exact :** `scipy.linalg.expm(-1j * H_matrix * t)`.
*   **Trotter (1er ordre) :** $U(t) \approx \left(\prod e^{-i c_k P_k t / n}\right)^n$.
*   **qDRIFT :** Norme $\lambda = \sum \vert c_k \vert$. Tirez un terme $P_k$ avec probabilité $p_k = \vert c_k \vert / \lambda$. L'évolution pour ce terme est $e^{-i \cdot \text{sign}(c_k) P_k \lambda t / N}$.
*   **Hybride :** Divisez vos termes, appliquez Trotter sur les majeurs, qDRIFT sur les mineurs, et multipliez les unitaires résultants.


In [ ]:
t = 1.0  # Temps d'évolution
n_steps = 10  # Nombre de pas (Trotter) ou d'échantillons (qDRIFT)
threshold = 0.1 # Seuil pour la méthode hybride

# TODO: Calculez la matrice unitaire exacte
# U_exact = ...

# TODO: Implémentez la construction de l'opérateur de Trotter
# U_trotter = ...

# TODO: Implémentez la construction via qDRIFT (échantillonnage aléatoire)
# U_qdrift = ...

# TODO: Implémentez la construction de la méthode Hybride
# U_hybrid = ...

## 5. Comparaison Quantitative
Évaluons la précision de nos méthodes d'approximation.

**Indices de réalisation :**
- Pour multiplier des matrices générées dynamiquement, initialisez l'opérateur identité avec `U_approx = np.eye(2**num_qubits, dtype=complex)` et mettez-le à jour avec `U_approx = term_matrix @ U_approx`.
- Utilisez la norme spectrale ou la norme de Frobenius pour l'erreur : `np.linalg.norm(U_exact - U_approx)`.
- Créez une boucle faisant varier le nombre de pas `n_steps` (ex: de 1 à 50).
- Calculez les erreurs pour Trotter, qDRIFT, et Hybride, et stockez-les dans des listes.
- Tracez l'évolution de ces erreurs en fonction du nombre de pas avec `plt.plot()` pour comparer la vitesse de convergence.

In [ ]:
steps_list = range(1, 50, 2)
erreurs_trotter = []
erreurs_qdrift = []
erreurs_hybride = []

# TODO: Bouclez sur steps_list pour calculer et stocker les erreurs
# for n in steps_list:
#     ...

# TODO: Tracez les courbes comparatives avec plt.plot()
# Pensez à ajouter une légende (plt.legend()) et des labels aux axes.

## 6. Intégration : Circuit avec Qubit de Contrôle (Hadamard Test)
Pour extraire des informations (comme dans l'algorithme d'estimation de phase quantique), nous devons souvent conditionner cette évolution à l'état d'un qubit auxiliaire (qubit de contrôle).

**Indices de réalisation :**
- Créez un `QuantumCircuit` avec 1 qubit de contrôle + $N$ qubits pour le système (dépendant du mapper choisi).
- Appliquez une porte Hadamard (`h`) sur le qubit de contrôle initial.
- Transformez votre matrice unitaire d'évolution en une porte quantique avec `UnitaryGate(U_exact)`.
- Rendez cette porte conditionnelle avec la méthode `.control(1)`.
- Ajoutez-la au circuit et refermez avec une seconde porte Hadamard sur le contrôle.

In [ ]:
num_system_qubits = 4 # Modifiez selon la taille de votre Hamiltonien
    
# TODO: Instanciez le circuit quantique (1 contrôle + système)
# qc = QuantumCircuit(...)

# TODO: Préparez le qubit de contrôle avec Hadamard
# qc.h(0)

# TODO: Créez la porte contrôlée à partir de la matrice d'évolution
# controlled_U = UnitaryGate(...).control(1)

# TODO: Appliquez la porte contrôlée sur les bons qubits du circuit
    
# TODO: Terminez par une porte Hadamard sur le contrôle et dessinez le circuit
# qc.draw('mpl')

## 7. Exploration : Mise à l'échelle selon la taille de la chaîne ($H_4$ ou $H_5$)

Maintenant que vous avez analysé $H_2$, utilisez votre fonction `get_h_chain_pauli_op` pour générer $H_4$ ou $H_5$.

**Questions d'exploration :**
1. Comment évolue le nombre de termes de Pauli par rapport à $H_2$ ?
2. Relancez l'analyse de convergence (Partie 5) sur cet Hamiltonien $H_4$ ou $H_5$. La méthode Hybride montre-t-elle un avantage encore plus marqué face à la méthode de Trotter classique au vu du nombre de petits coefficients qui apparaissent ?

In [ ]:
# TODO: Générez l'opérateur pour H3 (3 atomes)
# op_h3 = get_h_chain_pauli_op(3, 0.735, parity_mapper)

# TODO: Observez la distribution des coefficients (combien sont < threshold ?)

# TODO: Optionnel - Refaites le graphique d'erreur (Partie 5) pour H3